In [2]:

import pandas as pd
import plotly.express as px
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import webbrowser
import re

# ============================================================
# TASK 1: INTERACTIVE BUBBLE CHART
# App Size vs Average Rating | Bubble Size = Installs
# Dashboard available from 5 PM to before 7 PM IST
# ============================================================

# 1. Check and load both datasets
play_file = Path("Play Store Data.csv")
reviews_file = Path("User Reviews.csv")

if not play_file.exists():
    raise FileNotFoundError(
        "Play Store Data.csv was not found in the project folder."
    )

if not reviews_file.exists():
    raise FileNotFoundError(
        "User Reviews.csv was not found. Place it in the same folder "
        "as this notebook to apply the sentiment subjectivity filter."
    )

df = pd.read_csv(play_file)
reviews = pd.read_csv(reviews_file)

print("Play Store dataset loaded:", df.shape)
print("User Reviews dataset loaded:", reviews.shape)

# 2. Check required sentiment columns
if "Sentiment_Subjectivity" not in reviews.columns:
    raise ValueError(
        "User Reviews.csv must contain a Sentiment_Subjectivity column. "
        "Check the exact column names in your file."
    )

if "App" not in reviews.columns:
    raise ValueError("User Reviews.csv must contain an App column.")

# 3. Clean numeric values
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def size_to_mb(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip().upper()

    if value in ("", "NAN", "VARIES WITH DEVICE"):
        return float("nan")

    try:
        if value.endswith("M"):
            return float(value[:-1])
        elif value.endswith("K"):
            return float(value[:-1]) / 1024
        elif value.endswith("G"):
            return float(value[:-1]) * 1024
        else:
            return float(value)
    except ValueError:
        return float("nan")

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Reviews"] = clean_numeric(df["Reviews"])
df["Installs"] = clean_numeric(df["Installs"])
df["Size_MB"] = df["Size"].apply(size_to_mb)

# 4. Calculate average sentiment subjectivity per app
reviews["Sentiment_Subjectivity"] = pd.to_numeric(
    reviews["Sentiment_Subjectivity"], errors="coerce"
)

subjectivity = (
    reviews.dropna(subset=["App", "Sentiment_Subjectivity"])
    .groupby("App", as_index=False)["Sentiment_Subjectivity"]
    .mean()
)

print("Apps with subjectivity scores:", len(subjectivity))

# 5. Merge sentiment information into Play Store data
df = df.merge(subjectivity, on="App", how="inner")

# 6. Normalize category names and translate requested categories
category_translation = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}

allowed_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

df["Category"] = df["Category"].astype(str).str.strip().str.upper()

# 7. Apply all requested filters
# - Rating > 3.5
# - Requested categories only
# - Reviews > 500
# - App name must not contain S/s
# - Average sentiment subjectivity > 0.5
# - Installs > 50,000
# - Valid app size required for the x-axis
bubble_df = df[
    (df["Rating"] > 3.5) &
    (df["Category"].isin(allowed_categories)) &
    (df["Reviews"] > 500) &
    (~df["App"].astype(str).str.contains("s", case=False, na=False)) &
    (df["Sentiment_Subjectivity"] > 0.5) &
    (df["Installs"] > 50000) &
    (df["Size_MB"].notna())
].copy()

if bubble_df.empty:
    raise ValueError(
        "No apps matched all Task 7 filters. "
        "Check the data and required columns."
    )

# 8. Keep one record per app/category if duplicate app rows exist
bubble_df = (
    bubble_df.sort_values("Installs", ascending=False)
    .drop_duplicates(subset=["App", "Category"])
    .copy()
)

# Add translated display names
bubble_df["Display_Category"] = bubble_df.apply(
    lambda row: category_translation.get(
        row["Category"], row["Category"]
    ),
    axis=1
)

# Make GAME pink; other categories use separate colors
category_colors = {
    "GAME": "#FF69B4",
    "BEAUTY": "#9467BD",
    "BUSINESS": "#2CA02C",
    "COMICS": "#8C564B",
    "COMMUNICATION": "#17BECF",
    "DATING": "#E377C2",
    "ENTERTAINMENT": "#BCBD22",
    "SOCIAL": "#1F77B4",
    "EVENTS": "#FF7F0E"
}

bubble_df["Display_Color"] = bubble_df["Category"].map(category_colors)

# 9. Display summary
print("\nTask 7 filtering completed.")
print("Number of apps in chart:", len(bubble_df))
print("Categories represented:")
print(bubble_df["Category"].value_counts())

display(
    bubble_df[
        [
            "App", "Category", "Size_MB", "Rating",
            "Installs", "Reviews", "Sentiment_Subjectivity"
        ]
    ].head(10)
)

# 10. Create the interactive bubble chart
fig = px.scatter(
    bubble_df,
    x="Size_MB",
    y="Rating",
    size="Installs",
    color="Category",
    color_discrete_map=category_colors,
    hover_name="App",
    hover_data={
        "Display_Category": True,
        "Size_MB": ":.2f",
        "Rating": ":.2f",
        "Installs": ":,",
        "Reviews": ":,",
        "Sentiment_Subjectivity": ":.3f",
        "Category": False
    },
    size_max=55,
    title="Task 7: App Size vs Average Rating",
    labels={
        "Size_MB": "App Size (MB)",
        "Rating": "Average Rating",
        "Installs": "Number of Installs",
        "Display_Category": "Category",
        "Sentiment_Subjectivity": "Average Sentiment Subjectivity"
    }
)

fig.update_traces(
    marker=dict(opacity=0.75, line=dict(width=0.5, color="white"))
)

fig.update_layout(
    template="plotly_white",
    height=700,
    hovermode="closest",
    legend_title="App Category",
    margin=dict(l=70, r=40, t=90, b=70)
)

fig.show()

# 11. Create an HTML dashboard with a 5 PM–7 PM IST time gate
chart_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={"responsive": True, "displaylogo": False}
)

dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Task 7 - App Size and Rating Bubble Chart</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 20px;
    background: #f4f6f9;
    color: #222;
}}
.container {{
    max-width: 1400px;
    margin: auto;
    background: white;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
}}
#notice {{
    text-align: center;
    padding: 45px 20px;
    font-size: 20px;
}}
.note {{
    color: #555;
    font-size: 14px;
    text-align: center;
}}
</style>
</head>
<body>
<div class="container">
    <h2> Google Play Store Analytics</h2>
    <p class="note">
        Bubble size represents installs. GAME category is highlighted in pink.
        The subjectivity score is the average available review subjectivity
        for each app.
    </p>
    <div id="notice" style="display:none;">
        <h3>Dashboard currently unavailable</h3>
        <p>This dashboard is available only from 5:00 PM to before 7:00 PM IST.</p>
        <p id="currentTime"></p>
    </div>
    <div id="chart" style="display:none;">
        {chart_html}
    </div>
</div>
<script>
function checkDashboardTime() {{
    const parts = new Intl.DateTimeFormat("en-GB", {{
        timeZone: "Asia/Kolkata",
        hour: "2-digit",
        minute: "2-digit",
        hourCycle: "h23"
    }}).formatToParts(new Date());

    const hour = Number(parts.find(p => p.type === "hour").value);
    const minute = Number(parts.find(p => p.type === "minute").value);
    const currentMinutes = hour * 60 + minute;

    const allowed = currentMinutes >= 1020 && currentMinutes < 1140;

    document.getElementById("chart").style.display =
        allowed ? "block" : "none";
    document.getElementById("notice").style.display =
        allowed ? "none" : "block";

    document.getElementById("currentTime").textContent =
        "Current IST time: " +
        new Intl.DateTimeFormat("en-IN", {{
            timeZone: "Asia/Kolkata",
            hour: "2-digit",
            minute: "2-digit",
            second: "2-digit",
            hour12: true
        }}).format(new Date());
}}

checkDashboardTime();
setInterval(checkDashboardTime, 15000);
</script>
</body>
</html>
"""

output_file = Path("dashboard_task7.html")
output_file.write_text(dashboard_html, encoding="utf-8")

print("\nDashboard saved to:", output_file.resolve())
print("Display window: 5:00 PM to before 7:00 PM IST.")
print(
    "Current IST:",
    datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%I:%M:%S %p")
)

webbrowser.open(output_file.resolve().as_uri())
print("Dashboard opened in your default browser.")

Play Store dataset loaded: (10841, 13)
User Reviews dataset loaded: (64295, 5)
Apps with subjectivity scores: 865

Task 7 filtering completed.
Number of apps in chart: 24
Categories represented:
Category
GAME             11
DATING            5
COMMUNICATION     3
ENTERTAINMENT     2
BUSINESS          2
SOCIAL            1
Name: count, dtype: int64


,App,Category,Size_MB,Rating,Installs,Reviews,Sentiment_Subjectivity
455,Flow Free,GAME,11.0,4.3,100000000.0,1295606.0,0.520183
435,Cut the Rope FULL FREE,GAME,49.0,4.4,100000000.0,2123381.0,0.570271
459,Granny,GAME,59.0,4.5,50000000.0,1137267.0,0.556299
202,Amazon Prime Video,ENTERTAINMENT,24.0,4.2,50000000.0,411683.0,0.555556
1198,Dude Perfect 2,GAME,70.0,4.5,10000000.0,401648.0,0.535797
42,Google Primer,BUSINESS,18.0,4.4,10000000.0,62272.0,0.675000
561,1LINE – One Line with One Touch,GAME,10.0,4.6,10000000.0,214878.0,0.557315
550,Candy Bomb,GAME,20.0,4.4,10000000.0,42145.0,0.606239
1188,"CallApp: Caller ID, Blocker & Phone Call Recorder",COMMUNICATION,20.0,4.4,10000000.0,483782.0,0.506481
465,Draw In,GAME,73.0,4.2,10000000.0,46416.0,0.582500



Dashboard saved to: C:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task7.html
Display window: 5:00 PM to before 7:00 PM IST.
Current IST: 02:28:57 PM
Dashboard opened in your default browser.
